# Avaliação por Partidas Reais — ia-pontinhos-3-4 vs Minimax

Este notebook avalia o agente híbrido `ia-pontinhos-3-4` (Simbólico + CNN + Minimax) jogando partidas reais contra o Minimax puro em diferentes profundidades.

**Métricas coletadas:**
- Vitórias / Empates / Derrotas
- Distribuição de `co_situacao` e `co_acao` por jogada
- Qual User Story é mais utilizada no fechamento de caixas
- Tempos médios de decisão por User Story (comparáveis com o Minimax puro)

**Instrução Importante:** Certifique-se de selecionar o Kernel `.venv_tf` (ou Python do `.venv_tf`) no VS Code antes de rodar.

In [1]:
import sys, os

sys.path.insert(0, os.path.abspath('../..'))

from tqdm import tqdm

from gerador_dados.jogo_pontinhos.avaliador_hibrido_pontinhos import (
    avaliar_paralelo_hibrido,
    imprimir_relatorio_hibrido,
)
from gerador_dados.jogo_pontinhos.tipos_pontinhos_3_4 import (
    ConfiguracaoAgente,
    NivelDificuldade,
)

# -- Configuração do agente híbrido ------------------------------------
NIVEL           = NivelDificuldade.EXPERT
CAMINHO_MODELO  = '../../modelos/pontinhos_pequeno_profundidade_11.tflite'
SEED            = 42          # Reprodutibilidade (None = nao-deterministico)

CONFIG = ConfiguracaoAgente(
    nivel_dificuldade=NIVEL,
    caminho_modelo_cnn=CAMINHO_MODELO,
    seed_aleatoriedade=SEED,
)

# -- Parametros da avaliacao -------------------------------------------
TAMANHO         = 'pequeno'
N_PARTIDAS      = 200    # total por profundidade (100 hibrido 1o + 100 hibrido 2o)
PROFUNDIDADES   = [1, 3, 5, 6]
TIMER_MM_MS     = 0      # 0 = sem limite | ex: 5 = Minimax limitado a 5ms/jogada
MAX_WORKERS     = 14

print(f'Agente: ia-pontinhos-3-4 (nivel={NIVEL.value})')
print(f'Modelo CNN: {CAMINHO_MODELO}')
print(f'Profundidade Minimax interno: {CONFIG.profundidade_minimax}')
print(f'Aleatoriedade: {CONFIG.percentual_aleatoriedade:.0%}')
print(f'Partidas por profundidade: {N_PARTIDAS}')
print(f'Profundidades adversarias: {PROFUNDIDADES}')
if TIMER_MM_MS > 0:
    print(f'Timer ativo: Minimax adversario limitado a {TIMER_MM_MS}ms por jogada')
print()

Agente: ia-pontinhos-3-4 (nivel=expert)
Modelo CNN: ../../modelos/pontinhos_pequeno_profundidade_11.tflite
Profundidade Minimax interno: 3
Aleatoriedade: 0%
Partidas por profundidade: 200
Profundidades adversarias: [1, 3, 5, 6]



In [2]:
stats_list = []

if __name__ == '__main__':
    for prof in PROFUNDIDADES:
        nome = f'Minimax(p<={prof}, {TIMER_MM_MS}ms)' if TIMER_MM_MS > 0 else f'Minimax(p={prof})'

        with tqdm(total=N_PARTIDAS, desc=f'{nome:25s}', unit='partida', leave=True) as pbar:
            c = [0, 0, 0]  # [vitorias, empates, derrotas]

            def _cb(completed, total, result, _pbar=pbar, _c=c):
                if result['vencedor'] == 1:   _c[0] += 1
                elif result['vencedor'] == 0:  _c[1] += 1
                else:                          _c[2] += 1
                _pbar.update(completed - _pbar.n)
                _pbar.set_postfix({'V': _c[0], 'E': _c[1], 'D': _c[2]})

            s = avaliar_paralelo_hibrido(
                configuracao=CONFIG,
                prof=prof,
                nome_mm=nome,
                tamanho=TAMANHO,
                n_partidas=N_PARTIDAS,
                timer_mm_ms=TIMER_MM_MS,
                max_workers=MAX_WORKERS,
                progress_callback=_cb,
            )
        stats_list.append(s)

    imprimir_relatorio_hibrido(stats_list)

Minimax(p=6)             : 100%|██████████| 200/200 [17:52<00:00,  5.36s/partida, V=55, E=20, D=125]


AVALIAÇÃO POR PARTIDAS REAIS — ia-pontinhos-3-4 vs Minimax

  Adversário: Minimax(p=1)  (200 partidas)
  Vitórias Híbrido            186  ( 93.0%)
  Empates                      10  (  5.0%)
  Derrotas Híbrido              4  (  2.0%)
  Tempo médio Híbrido:      59.92 ms/jogada
  Tempo médio Minimax(p=1): 0.2 ms/jogada
  Híbrido é 0× mais rápido

  ── Decisões do Híbrido (3822 jogadas) ───────────────────────────────
  User Story / Situação                    Jogadas       %  Tempo médio
  ────────────────────────────────────────────────────────────────────
  US1 — Captura Segura (grau-3)              1910   50.0%      0.20 ms
  US2 — Exceção do Sacrifício                   0    0.0%      0.00 ms
  US3+4 — CNN + Minimax Validado             1912   50.0%    118.48 ms
  Timeouts (fallback)                           0    0.0%            —
  ────────────────────────────────────────────────────────────────────

  ── Fechamento de Caixas (1930 caixas em 1910 jogadas) ──────────────────
  Us

## Detalhamento: Distribuicao de co_situacao e co_acao por profundidade

In [3]:
import pandas as pd

# -- Tabela: Distribuicao de co_situacao --------------------------------
rows_sit = []
for s in stats_list:
    for sit, info in s.get('dist_situacao', {}).items():
        rows_sit.append({
            'Adversario': s['adversario'],
            'co_situacao': sit,
            'Jogadas': info['count'],
            '%': f"{info['pct']:.1f}",
            'Tempo medio (ms)': f"{info['tempo_medio_ms']:.2f}",
        })

if rows_sit:
    df_sit = pd.DataFrame(rows_sit)
    print('=== Distribuicao de co_situacao por adversario ===')
    print(df_sit.to_string(index=False))
    print()

# -- Tabela: Distribuicao de co_acao ------------------------------------
rows_acao = []
for s in stats_list:
    for acao, info in s.get('dist_acao', {}).items():
        rows_acao.append({
            'Adversario': s['adversario'],
            'co_acao': acao,
            'Jogadas': info['count'],
            '%': f"{info['pct']:.1f}",
            'Tempo medio (ms)': f"{info['tempo_medio_ms']:.2f}",
        })

if rows_acao:
    df_acao = pd.DataFrame(rows_acao)
    print('=== Distribuicao de co_acao por adversario ===')
    print(df_acao.to_string(index=False))
    print()

# -- Tabela: Fechamento de caixas por User Story ------------------------
rows_fecha = []
for s in stats_list:
    for acao, info in s.get('dist_acao_fechamento', {}).items():
        rows_fecha.append({
            'Adversario': s['adversario'],
            'co_acao (fechamento)': acao,
            'Jogadas': info['count'],
            '% (fechamentos)': f"{info['pct']:.1f}",
            'Tempo medio (ms)': f"{info['tempo_medio_ms']:.2f}",
        })

if rows_fecha:
    df_fecha = pd.DataFrame(rows_fecha)
    print('=== Fechamento de Caixas por User Story ===')
    print(df_fecha.to_string(index=False))

=== Distribuicao de co_situacao por adversario ===
  Adversario    co_situacao  Jogadas    % Tempo medio (ms)
Minimax(p=1)         tatica     1912 50.0           118.48
Minimax(p=1) captura_segura     1910 50.0             0.20
Minimax(p=3)         tatica     1940 58.9           124.03
Minimax(p=3) captura_segura     1355 41.1             0.25
Minimax(p=5)         tatica     1966 64.4           119.01
Minimax(p=5) captura_segura     1089 35.6             0.30
Minimax(p=6)         tatica     1988 67.9           115.91
Minimax(p=6) captura_segura      939 32.1             0.31

=== Distribuicao de co_acao por adversario ===
  Adversario        co_acao  Jogadas    % Tempo medio (ms)
Minimax(p=1)  cnn_e_minimax     1912 50.0           118.48
Minimax(p=1) captura_gulosa     1910 50.0             0.20
Minimax(p=3)  cnn_e_minimax     1940 58.9           124.03
Minimax(p=3) captura_gulosa     1355 41.1             0.25
Minimax(p=5)  cnn_e_minimax     1966 64.4           119.01
Minimax(p=5) cap

## Resumo Consolidado (vitorias vs profundidade)

In [4]:
import pandas as pd

rows = []
for s in stats_list:
    rows.append({
        'Adversario': s['adversario'],
        'Partidas': s['partidas'],
        'Vitorias': s['vitorias_hibrido'],
        'Empates': s['empates'],
        'Derrotas': s['derrotas_hibrido'],
        '% Vitoria': f"{s['pct_vitorias']:.1f}%",
        'Tempo Hibrido (ms)': f"{s['tempo_hibrido_ms']:.2f}",
        'Tempo MM (ms)': f"{s['tempo_mm_ms']:.1f}",
        'Fator Velocidade': f"{s['fator_velocidade']:.0f}x",
    })

if rows:
    df = pd.DataFrame(rows)
    print('=== Resumo: ia-pontinhos-3-4 vs Minimax ===')
    print(df.to_string(index=False))

=== Resumo: ia-pontinhos-3-4 vs Minimax ===
  Adversario  Partidas  Vitorias  Empates  Derrotas % Vitoria Tempo Hibrido (ms) Tempo MM (ms) Fator Velocidade
Minimax(p=1)       200       186       10         4     93.0%              59.92           0.2               0x
Minimax(p=3)       200       121       31        48     60.5%              75.63          84.2               1x
Minimax(p=5)       200        83       34        83     41.5%              81.30        1455.2              18x
Minimax(p=6)       200        55       20       125     27.5%              83.81        4545.4              54x


## Resultados Anteriores (CNN Pura)

Os resultados abaixo sao das execucoes anteriores com a **CNN pura** (sem pipeline simbolico), mantidos para referencia e comparacao.

### CNN Profundidade 6

In [5]:
# ========================================================================
# AVALIACAO POR PARTIDAS REAIS - CNN vs Minimax
# **CNN TREINADA COM TABULEIROS GERADOS DE FORMA ALEATORIA**
# ========================================================================
#
#   Adversario: Minimax(p=1)  (200 partidas)
#   Vitorias CNN          192  ( 96.0%)
#   Empates                 4  (  2.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo medio CNN:  0.06 ms/jogada
#   Tempo medio Minimax(p=1): 0.2 ms/jogada
#   CNN e 3x mais rapida
#
#   Adversario: Minimax(p=3)  (200 partidas)
#   Vitorias CNN          105  ( 52.5%)
#   Empates                26  ( 13.0%)
#   Derrotas CNN           69  ( 34.5%)
#   Tempo medio CNN:  0.10 ms/jogada
#   Tempo medio Minimax(p=3): 82.4 ms/jogada
#   CNN e 847x mais rapida
#
#   Adversario: Minimax(p=5)  (200 partidas)
#   Vitorias CNN           44  ( 22.0%)
#   Empates                13  (  6.5%)
#   Derrotas CNN          143  ( 71.5%)
#   Tempo medio CNN:  0.13 ms/jogada
#   Tempo medio Minimax(p=5): 1639.1 ms/jogada
#   CNN e 12519x mais rapida
#
#   Adversario: Minimax(p=6)  (200 partidas)
#   Vitorias CNN            3  (  1.5%)
#   Empates                10  (  5.0%)
#   Derrotas CNN          187  ( 93.5%)
#   Tempo medio CNN:  0.15 ms/jogada
#   Tempo medio Minimax(p=6): 5107.0 ms/jogada
#   CNN e 34794x mais rapida
#
# ========================================================================

### CNN Profundidade 7

In [6]:
# ========================================================================
# AVALIACAO POR PARTIDAS REAIS - CNN vs Minimax
# ========================================================================
#
#   Adversario: Minimax(p=1)  (200 partidas)
#   Vitorias CNN          184  ( 92.0%)
#   Empates                12  (  6.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo medio CNN:  0.09 ms/jogada
#   Tempo medio Minimax(p=1): 0.2 ms/jogada
#   CNN e 2x mais rapida
#
#   Adversario: Minimax(p=3)  (200 partidas)
#   Vitorias CNN          120  ( 60.0%)
#   Empates                54  ( 27.0%)
#   Derrotas CNN           26  ( 13.0%)
#   Tempo medio CNN:  0.11 ms/jogada
#   Tempo medio Minimax(p=3): 80.6 ms/jogada
#   CNN e 711x mais rapida
#
#   Adversario: Minimax(p=5)  (200 partidas)
#   Vitorias CNN           89  ( 44.5%)
#   Empates                59  ( 29.5%)
#   Derrotas CNN           52  ( 26.0%)
#   Tempo medio CNN:  0.13 ms/jogada
#   Tempo medio Minimax(p=5): 1627.1 ms/jogada
#   CNN e 12419x mais rapida
#
#   Adversario: Minimax(p=6)  (200 partidas)
#   Vitorias CNN           80  ( 40.0%)
#   Empates                46  ( 23.0%)
#   Derrotas CNN           74  ( 37.0%)
#   Tempo medio CNN:  0.14 ms/jogada
#   Tempo medio Minimax(p=6): 5820.3 ms/jogada
#   CNN e 41782x mais rapida
#
# ========================================================================